This notebook describes the steps for the Amnesty International's naming and shaming example (third application). Data is from Strezhnev, Kelley and Simmons (2021). Prepared by Annamaria Prati and Yehu Chen.

# Setup

Let's set ourselves up. Load the required libraries. Like before, note that `torch` should be version 2.6.0 and `gpytorch` should be version 1.8.1. Set the seed and the default data type to be `float64`. 

In [1]:
# load gpytoch and other libraries
import torch
import numpy as np
import pandas as pd
import gpytorch
from scipy.stats import norm
from typing import Optional, Tuple
from matplotlib import pyplot as plt
from gpytorch.means import LinearMean
from gpytorch.likelihoods import GaussianLikelihood
from gpytorch.kernels import ScaleKernel, RBFKernel
import statsmodels.formula.api as smf

# set a random seed so results are consistent each time you run the code
torch.manual_seed(12345)

# Data Preparation

As discussed in the main text, we will focus on the Physical Integrity Rights Index (PIRI) for our application. So next, let's load the PIRI dataset and clean it for analyses. 

We will have to remove New Zealand and the Netherlands since they have all zero values, since GPR would not be able to model these non-changing values. 

We will create lookup dictionaries for countries and years since we need numeric inputs for our GPR model. 

We then create a design matrix filled with zeros with the same number of rows as our dataset and 10 columns for our 10 features. We will then fill in each column with the features from either our dictionary or directly from our dataset: year, country, whether or not Amnesty International shamed that country in that year, whether that country had ratified the Convention Against Torture, whether the country had ratified the International Covenant on Civil and Political Rights, whether the country is a democracy, GDP per capita (logged), population (logged), whether the country is experiencing a civil war, and whether the country is at war. 

The dependent variable is the leading value of of that country's PIRI index, meaning the year after Amnesty International may have named/shamed that country. 

We then will calculate the mean $y$ (leading PIRI score) for each country.  We will use these means as a fixed effect initialization for each country’s mean function, providing a better startin point for optimization than if we were to start all countries at zero. This is like how we would include country fixed effects in ordinary least squares, meaning each country gets its own intercept. 

We will also cast to double precision for `gpytorch`.

In [2]:
def load_PIRI_data(): 
    # read data 
    data = pd.read_csv("./data/hb_data_complete.csv", index_col=[0])
    
    # dropping countries with all zero PIRI values
    data = data.loc[~data['country'].isin(['N-ZEAL','NETHERL'])]

    # creating our lookup dictionaries for countries and years
    countries = sorted(data.country.unique())
    years = data.year.unique()
    n = len(countries)
    m = len(years)
    country_dict = dict(zip(countries, range(n)))
    year_dict = dict(zip(years, range(m)))

    # building the design matrix
    X = torch.zeros(data.shape[0], 10)

    # filling in the design matrix with features
        # Columns:
        # 0: year number
        # 1: country id
        # 2: AIShame (treatment indicator)
        # 3: cat_rat
        # 4: ccpr_rat
        # 5: democratic
        # 6: log(gdppc)
        # 7: log(pop)
        # 8: Civilwar2
        # 9: War
    X[:,0] = torch.as_tensor(list(map(year_dict.get, data.year)))
    X[:,1] = torch.as_tensor(list(map(country_dict.get, data.country)))
    X[:,2] = torch.as_tensor(data.AIShame.to_numpy())
    X[:,3] = torch.as_tensor(data.cat_rat.to_numpy())
    X[:,4] = torch.as_tensor(data.ccpr_rat.to_numpy())
    X[:,5] = torch.as_tensor(data.democratic.to_numpy())
    X[:,6] = torch.as_tensor(data.log_gdppc.to_numpy())
    X[:,7] = torch.as_tensor(data.log_pop.to_numpy())
    X[:,8] = torch.as_tensor(data.Civilwar2.to_numpy())
    X[:,9] = torch.as_tensor(data.War.to_numpy())

    # building the dependent variable PIRILead1 (PIRI in the following year)
    Y = torch.as_tensor(data.PIRILead1.to_numpy()).double()

    # compute unit means for each country
    unit_means = torch.zeros(n,)
    for i in range(n):
        unit_means[i] = Y[X[:,1] == i].mean()

    return X.double(), Y.double(), unit_means.double(), data, countries, years

train_x, train_y, unit_means, data, countries, years = load_PIRI_data()

We will also do another new step in this example: to help us provide more reasonable starting values for the GP, we will extract the coefficients from a two-way fixed effects regression using ordinary least squares and use those as the linear mean weights later. This is more efficient than letting the GP randomly initialize the linear mean's weights since it doesn't have to "search" as much. We can use the contemporaneous relationship between PIRI and the coefficients, since this gives a cleaner baseline (no "peaking ahead"). 

For technical reasons, the `smf.ols()` portion should be run in a separate notebook. It is included here as raw text.

Now we read in the coefficients from the previously run OLS model.

In [3]:
# read in csv of coefficients 
coef_df = pd.read_csv("./results/PIRI_coefficients.csv")
# create weights
x_weights = coef_df["coefficient"]
covariate_names = ["AIShame", "cat_rat", "ccpr_rat",
                   "democratic", "log_gdppc", "log_pop",
                   "Civilwar2", "War"]

# 3. Extract coefficients in correct order
x_weights_list = [coef_df.loc[coef_df['variable'] == name, 'coefficient'].values[0] 
                  for name in covariate_names]

# 4. Convert to tensor and reshape for LinearMean
x_weights_tensor = torch.tensor(x_weights_list, dtype=torch.float32).reshape(-1, 1)  # shape [8,1]

# Model Specification

Like the previous examples, we will conceptualize the problem as an additive combination of processes:

$$y_{it} = \delta(a_{i,t-1}) + f(\mathbf x_{i,t-1}) + h_i(t) + \varepsilon_{i,t}$$

The first component is modeling Amnesty International's shaming practices, using the SE kernel as the covariance function: 

\begin{align}
    \delta(\mathbf{a}) &\sim GP(\mathbf{0},\mathbf{K}_a)
\end{align}

The second component is modeling the confounding covariates. We apply a GP prior with a zero mean function, and we create a kernel by adding two sets of SE kernels together: one for continuous covariates (subscript with $c$) and one for binary covariates (subscript with $b$). Mathematically, this yields:

\begin{eqnarray}
    f  \sim \mathcal{GP}\big(\mathbf 0, \mathbf{K}_f \big),\\
    \mathbf{K}_f= \mathbf{K}^{SE}(\mathbf{x_c})+ \mathbf{K}^{SE}(\mathbf{x_b})
\end{eqnarray}

The third component is a constant mean function (see more below, but essentially models country fixed effects) with an SE kernel and time as an input. In this case, we can use a spatiotemporal kernel that interacts an SE kernel for country an an SE kernel for year.
\begin{eqnarray}
    h_i(t)  \sim \mathcal{GP}\big(\boldsymbol \mu_{h[i]}(t), \mathbf K_h\big), \text{ where}\\
    \boldsymbol \mu_{h[i]}(t) = \alpha_{[i]}\\
    \mathbf K_h = \mathbf{K}^{SE}(\mathbf{x_i}) \times \mathbf{K}^{SE}(\mathbf{x_t})
\end{eqnarray}

The final component is Gaussian error.

When we combine the pieces, the complete model is
\begin{eqnarray}
\mathbf y \sim \mathcal{MVN}(\boldsymbol \mu_y, \mathbf{K}_y), \text{ where}\\
\boldsymbol \mu_y = \boldsymbol \mu_{h[i]} \text{, and}\\
\mathbf{K}_y = \mathbf{K}_{\delta} + \mathbf{K}_{f} + \mathbf{K}_h + \mathbf{I}\sigma^2_{\text{noise}}.
\end{eqnarray}



# Constructing the Model

Let's start constructing our model. First, let's define a class called `ConstantVectorMean`, which will let our GP model have one baseline mean per country (like in a fixed effects regression). This means that the GP kernel does not have to waste resources modeling persistent country-level differences, our `ConstantVectorMean` will absorb those differences. We do this by creating a lookup-table mean function: first, allow for `d` number of categories (e.g. the number of countries) and create a learnable parameter called `constantvector` of shape `(d,)`. In the `forward`, we take our input categorical index for each country, convert those indices into an integer and look that up in `constantvector`, and return a vector where each entry is the corresponding country's constant mean. 

In [4]:
class ConstantVectorMean(gpytorch.means.mean.Mean):
    def __init__(self, d=1, prior=None, batch_shape=torch.Size(), **kwargs):
        super().__init__()
        self.batch_shape = batch_shape
        self.register_parameter(name="constantvector",\
                 parameter=torch.nn.Parameter(torch.zeros(*batch_shape, d)))
        if prior is not None:
            self.register_prior("mean_prior", prior, "constantvector")

    def forward(self, input):
        return self.constantvector[input.int().reshape((-1,)).tolist()]

We again define the MaskMean class to apply a mean function only to selected input dimensions. 

In [5]:
class MaskMean(gpytorch.means.mean.Mean):
    def __init__(
        self,
        base_mean: gpytorch.means.mean.Mean,
        active_dims: Optional[Tuple[int, ...]] = None,
        **kwargs,
    ):
        super().__init__()
        if active_dims is not None and not torch.is_tensor(active_dims):
            active_dims = torch.tensor(active_dims, dtype=torch.long)
        self.active_dims = active_dims
        self.base_mean = base_mean
    
    def forward(self, x, **params):
        return self.base_mean.forward(x.index_select(-1, self.active_dims), **params)


And we can now put everything together in the `GPModel` class. This one can be an ExactGP since our dataset is smaller.

In [6]:
class GPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super().__init__(train_x, train_y, likelihood)
        # constant country-level mean
        self.mean_module = MaskMean(active_dims=1, \
               base_mean=ConstantVectorMean(d=train_x[:,1].unique().size()[0]))
        # linear mean for covariates
        self.x_mean_module = MaskMean(active_dims=[2,3,4,5,6,7,8,9], base_mean=LinearMean(input_size=8, bias=False))
        # year kernel * country kernel
        self.unit_covar_module = ScaleKernel(RBFKernel(active_dims=0)*RBFKernel(active_dims=1))
        # cov for continuous covariates
        self.x_covar_module = torch.nn.ModuleList([ScaleKernel(RBFKernel(\
            active_dims=(i))) for i in [6,7]])
        # cov for binary covariates
        self.binary_covar_module = torch.nn.ModuleList([ScaleKernel(RBFKernel(\
            active_dims=(i))) for i in [3,4,5,8,9]])
        # cov for treatment effect
        self.effect_covar_module = ScaleKernel(RBFKernel(active_dims=2))

    def forward(self, x):
        # total mean
        mean_x = self.mean_module(x) + self.x_mean_module(x)
        # adding all the covariances
        unit_covar_x = self.unit_covar_module(x) 
        effect_covar_x = self.effect_covar_module(x)
        covar_x = unit_covar_x + effect_covar_x
        for i, _ in enumerate(self.x_covar_module):
            covar_x += self.x_covar_module[i](x)
        for i, _ in enumerate(self.binary_covar_module):
            covar_x += self.binary_covar_module[i](x)
            
        # returning the posterior
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

We can now define our likelihood and model, as well as initialize our model parameters (which partly utilize the two-way fixed effects weights calculated earlier).

# Training the Model

We again switch the model and likelihood into "training" mode. We again use the Adam optimizer with a learning rate of 0.1 In this case, we are going to collect all the learnable parameters into a set, remove the ones we do not want optimized (freeze them), and then converts them back to a list for PyTorch. We will also save the results -- this could be useful if we wanted to do predictions later with a hold out dataset.

In [7]:
likelihood = GaussianLikelihood()
model = GPModel(train_x, train_y, likelihood).double()

# initialize model parameters
hypers = {
    'mean_module.base_mean.constantvector': unit_means,
    'x_mean_module.base_mean.weights': torch.tensor(x_weights_tensor),
    'likelihood.noise_covar.noise': torch.tensor(0.25),
    'unit_covar_module.base_kernel.kernels.0.lengthscale': torch.tensor(6),
    'unit_covar_module.base_kernel.kernels.1.lengthscale': torch.tensor(0.01),
    'unit_covar_module.outputscale': torch.tensor(4),
    # 'group_covar_module.base_kernel.lengthscale': torch.tensor(12),
    # 'group_covar_module.outputscale': torch.tensor(4),
    'x_covar_module.0.outputscale': torch.tensor(1),
    'x_covar_module.1.outputscale': torch.tensor(1),
    'binary_covar_module.0.base_kernel.lengthscale': torch.tensor(0.01),
    'binary_covar_module.1.base_kernel.lengthscale': torch.tensor(0.01),
    'binary_covar_module.2.base_kernel.lengthscale': torch.tensor(0.01),
    'binary_covar_module.3.base_kernel.lengthscale': torch.tensor(0.01),
    'binary_covar_module.4.base_kernel.lengthscale': torch.tensor(0.01),
    'binary_covar_module.0.outputscale': torch.tensor(1),
    'binary_covar_module.1.outputscale': torch.tensor(1),
    'binary_covar_module.2.outputscale': torch.tensor(1),
    'binary_covar_module.3.outputscale': torch.tensor(1),
    'binary_covar_module.4.outputscale': torch.tensor(1),
    'effect_covar_module.base_kernel.lengthscale': torch.tensor(0.01),
    'effect_covar_module.outputscale': torch.tensor(1)
}    

model = model.initialize(**hypers)

C:\Users\miame\AppData\Local\Temp\ipykernel_22992\2510135790.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'x_mean_module.base_mean.weights': torch.tensor(x_weights_tensor),


In [8]:
# train model
model.train()
likelihood.train()

# freeze length scale in the country component in unit covar
# freeze constant unit means
all_params = set(model.parameters())
final_params = list(all_params - \
            {model.unit_covar_module.base_kernel.kernels[1].raw_lengthscale, \
            model.binary_covar_module[0].base_kernel.raw_lengthscale,
            model.binary_covar_module[1].base_kernel.raw_lengthscale,
            model.binary_covar_module[2].base_kernel.raw_lengthscale,
            model.binary_covar_module[3].base_kernel.raw_lengthscale,
            model.binary_covar_module[4].base_kernel.raw_lengthscale,
            model.effect_covar_module.base_kernel.raw_lengthscale})
optimizer = torch.optim.Adam(final_params, lr=0.1)

# "loss" for GPs: marginal log likelihood
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

training_iter = 100
for i in range(training_iter):
    # Zero gradients from previous iteration
    optimizer.zero_grad()
    # Output from model
    output = model(train_x)
    # Calc loss and backprop gradients
    loss = -mll(output, train_y)
    loss.backward()
    if i % 10 == 0:
        print('Iter %d/%d - Loss: %.3f '  % (
            i , training_iter, loss.item()
        ))
    optimizer.step()

# saving the trained model
torch.save(model.state_dict(), "PIRI_GPR_model.pth")

C:\Users\miame\anaconda3\Lib\site-packages\gpytorch\functions\_pivoted_cholesky.py:118: UserWarning: torch.triangular_solve is deprecated in favor of torch.linalg.solve_triangularand will be removed in a future PyTorch release.
torch.linalg.solve_triangular has its arguments reversed and does not return a copy of one of the inputs.
X = torch.triangular_solve(B, A).solution
should be replaced with
X = torch.linalg.solve_triangular(A, B). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\BatchLinearAlgebra.cpp:2259.)
  [L, torch.triangular_solve(Krows[..., m:, :].transpose(-1, -2), L, upper=False)[0].transpose(-1, -2)],


Iter 0/100 - Loss: 2.252 
Iter 10/100 - Loss: 1.738 
Iter 20/100 - Loss: 1.637 
Iter 30/100 - Loss: 1.617 
Iter 40/100 - Loss: 1.602 
Iter 50/100 - Loss: 1.582 
Iter 60/100 - Loss: 1.574 
Iter 70/100 - Loss: 1.570 
Iter 80/100 - Loss: 1.563 
Iter 90/100 - Loss: 1.564 


# Evaluating the Model

We load our saved model. Again, we freeze our learned hyperparameters by switching the model and likelihood into "evaluation" mode and obtain our posterior predictions for both sets of inputs. We store them to a .csv file.

In [9]:
# load the trained model
model.load_state_dict(torch.load('PIRI_GPR_model.pth'))

# set model and likelihood to evaluation mode
model.eval()
likelihood.eval()

# posterior predictions
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    out = model(train_x)
    mu_f = out.mean.numpy()
    lower, upper = out.confidence_region()

# store results
results = pd.DataFrame({"gpr_mean":mu_f})
results['true_y'] = train_y
results['gpr_lwr'] = lower
results['gpr_upr'] = upper
results['year'] = years[train_x[:,0].numpy().astype(int)]
results['country'] = [countries[i] for i in train_x[:,1].numpy().astype(int)]
results.to_csv("./results/PIRI_fitted_gpr.csv",index=False) #save to file


C:\Users\miame\anaconda3\Lib\site-packages\gpytorch\models\exact_gp.py:273: GPInputWarning: The input matches the stored training data. Did you forget to call model.train()?
  warnings.warn(


We can also calculate the ATE, measures of uncertainty, and goodness of fit.

In [10]:
# print RMSE
RMSE = np.square((out.mean - train_y).detach().numpy()).mean()**0.5
print(RMSE)

# calculating the ATE and standard error
# copy training tesnor to test tensors and set AIshame to 1 and 0
test_x1 = train_x.clone().detach().requires_grad_(False)
test_x1[:,2] = 1
test_x0 = train_x.clone().detach().requires_grad_(False)
test_x0[:,2] = 0

# in eval mode the forward() function returns posterioir
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    out1 = likelihood(model(test_x1))
    out0 = likelihood(model(test_x0))

# compute ATE and its uncertainty
effect = out1.mean.numpy().mean() - out0.mean.numpy().mean()
effect_std = np.sqrt((out1.mean.numpy()[test_x1[:,2] == 1].mean()\
                      +out0.mean.numpy()[test_x1[:,2] == 1].mean())) / np.sqrt(train_x[test_x1[:,2] == 1].size()[0])
BIC = (2+4+6+1)*torch.log(torch.tensor(train_x.size()[0])) + 2*loss*train_x.size()[0]
print("ATE: {:0.3f} +- {:0.3f}\n".format(effect, effect_std))
print("model evidence: {:0.3f} \n".format(-loss*train_x.size()[0]))
print("BIC: {:0.3f} \n".format(BIC))

0.873814996513744
ATE: 0.060 +- 0.054

model evidence: -3334.051 

BIC: 6767.775 

